# Improved Forecasting Plan Implementation

Full Gibbs AM-BVAR-SV forecast workflow for core PCE inflation and components.

This notebook estimates a five-variable Bayesian VAR with stochastic volatility and an adaptive Minnesota-style prior. The model includes ECI wage growth, services ex-housing inflation, housing inflation, core goods inflation, and aggregate core PCE inflation, ordered in that sequence, and uses four lags. The reduced-form innovations are represented as `v_t = A^-1 Lambda_t^0.5 epsilon_t`, where `A` is lower triangular with unit diagonal and `Lambda_t` is diagonal with latent time-varying variances. The log variances follow random walks with innovations distributed `N(0, Phi)`, where `Phi` is a full covariance matrix, allowing volatility innovations to be correlated across variables.

The VAR coefficients have a zero-mean adaptive Minnesota-style prior with separate own-lag and cross-lag shrinkage hyperparameters. The model is estimated with a Gibbs sampler that alternately samples VAR coefficients, the lower-triangular `A` matrix, latent stochastic-volatility paths, the full covariance matrix `Phi`, and adaptive shrinkage hyperparameters. Forecasts are computed from the posterior predictive distribution by simulating future volatility innovations, structural shocks, reduced-form shocks, and VAR outcomes recursively over the forecast horizon.

Raw index levels begin in **2001Q1**. Because the model variables are annualized quarterly log changes, **2001Q1 is used only as an index-level anchor** and the transformed model sample begins in **2001Q2**.


## What This Notebook Produces

- Table 1: Data definitions and line-description checks
- Table 2: Recursive RMSEs and relative RMSEs versus full Gibbs AM-BVAR-SV
- Table 3: Final annualized QoQ log-growth forecasts
- Table 4: Final implied index-level forecasts
- Optional Table 5: AM-BVAR-SV posterior forecast intervals for growth rates and index levels
- Full Gibbs diagnostics: adaptive-shrinkage traces, `A`/`Phi` summaries, and latent-volatility plots
- Wage-growth extension tables: five-variable AM-BVAR-SV versus four-variable no-wage AM-BVAR-SV using the same Gibbs sampler
- Wage-growth extension figures: relative RMSE heatmap, cumulative squared-error differences, and forecast comparisons

The notebook writes reproducible CSV and PNG outputs under `data/` and `outputs/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

from forecasting_workflow import *

pd.options.display.float_format = "{:,.3f}".format

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
KEY_FILE = PROJECT_DIR / "API Keys.txt"

# Runtime controls. Full Gibbs recursive evaluation is much heavier than the earlier shortcut.
# The defaults below are practical development settings; increase n_iter and retained draws for publication runs.
FORCE_RECOMPUTE_DATA = True
RUN_RECURSIVE_EVALUATION = True
FORCE_RECOMPUTE_RECURSIVE = True

RANDOM_SEED = 20260519
AM_BVAR_N_ITER_RECURSIVE = DEFAULT_GIBBS_N_ITER_RECURSIVE
AM_BVAR_BURN_RECURSIVE = DEFAULT_GIBBS_BURN_RECURSIVE
AM_BVAR_THIN_RECURSIVE = DEFAULT_GIBBS_THIN_RECURSIVE
AM_BVAR_TRAJECTORIES_RECURSIVE = DEFAULT_GIBBS_TRAJECTORIES_RECURSIVE
AM_BVAR_N_ITER_FINAL = DEFAULT_GIBBS_N_ITER_FINAL
AM_BVAR_BURN_FINAL = DEFAULT_GIBBS_BURN_FINAL
AM_BVAR_THIN_FINAL = DEFAULT_GIBBS_THIN_FINAL
AM_BVAR_TRAJECTORIES_FINAL = DEFAULT_GIBBS_TRAJECTORIES_FINAL
UCSV_PARTICLES_RECURSIVE = 500
UCSV_DRAWS_RECURSIVE = 500
UCSV_PARTICLES_FINAL = 2500
UCSV_DRAWS_FINAL = 5000
FINAL_FORECAST_HORIZON = 7
UCSV_PARTICLES_FIT = 1000
PLOT_VARIABLE = "core_pce"
PLOT_HORIZON = 1

RUN_WAGE_EXTENSION = True
FORCE_RECOMPUTE_WAGE_EXTENSION = True
RUN_WAGE_ROBUSTNESS_2014 = True
WAGE_EXTENSION_FIRST_ORIGIN = FIRST_FORECAST_ORIGIN
WAGE_EXTENSION_ROBUSTNESS_ORIGIN = ROBUSTNESS_FORECAST_ORIGIN
WAGE_AM_BVAR_N_ITER_RECURSIVE = AM_BVAR_N_ITER_RECURSIVE
WAGE_AM_BVAR_BURN_RECURSIVE = AM_BVAR_BURN_RECURSIVE
WAGE_AM_BVAR_THIN_RECURSIVE = AM_BVAR_THIN_RECURSIVE
WAGE_AM_BVAR_TRAJECTORIES_RECURSIVE = AM_BVAR_TRAJECTORIES_RECURSIVE
WAGE_CUMULATIVE_HORIZONS = [4, 8]
WAGE_FORECAST_PLOT_VARIABLES = ["services_ex_housing", "core_pce"]
WAGE_FORECAST_PLOT_HORIZONS = [4, 8]

print("Project directory:", PROJECT_DIR)
print("Forecast horizons:", HORIZONS)
print("First recursive forecast origin:", FIRST_FORECAST_ORIGIN)
print("Recursive AM-BVAR-SV Gibbs settings:", AM_BVAR_N_ITER_RECURSIVE, "iterations,", AM_BVAR_BURN_RECURSIVE, "burn, thin", AM_BVAR_THIN_RECURSIVE)
print("Final AM-BVAR-SV Gibbs settings:", AM_BVAR_N_ITER_FINAL, "iterations,", AM_BVAR_BURN_FINAL, "burn, thin", AM_BVAR_THIN_FINAL)


## 1. Download Raw Index Levels

The BEA and FRED API keys are loaded from `API Keys.txt` but are never printed. BEA Table 2.4.4U is requested through the BEA `NIUnderlyingDetail` dataset as table `U20404`; BEA Table 2.3.4 is requested through the BEA `NIPA` dataset as table `T20304`.

In [ ]:
api_keys = load_api_keys(KEY_FILE)
print("Loaded API keys for:", ", ".join(sorted(api_keys)))

index_levels, data_definitions = load_or_download_data(
    PROJECT_DIR,
    api_keys,
    force_recompute=FORCE_RECOMPUTE_DATA,
)

print("Raw index levels shape:", index_levels.shape)
print("Raw index sample begins after constraint:", RAW_INDEX_START)
print("Latest common complete index quarter:", index_levels.dropna().index.max())

table_1_cols = [
    "variable_name",
    "source_table_or_series",
    "line_item_or_series_id",
    "downloaded_description",
    "description_check",
    "raw_index_start_date_after_constraint",
    "first_transformed_observation",
    "downloaded_latest_date",
    "transformation",
]
display(data_definitions[table_1_cols])
display(index_levels.rename(columns=INDEX_LABELS).tail())

## 2. Transform to Annualized Quarterly Log Changes

For every index variable, the model variable is:

`400 * log(index_t / index_t-1)`

No 2000Q4 observation is imputed. The first transformed observation is 2001Q2.

In [ ]:
common_growth, common_index_levels = transform_index_levels(index_levels)
common_growth.to_csv(DATA_DIR / "transformed_growth_rates.csv")

latest_common_quarter = common_growth.index.max()
print("Modeled transformed sample:", common_growth.index.min(), "to", latest_common_quarter)
print("Usable transformed observations:", len(common_growth))

display(common_growth.rename(columns=VAR_LABELS).head())
display(common_growth.rename(columns=VAR_LABELS).tail())

## 3. Recursive Out-of-Sample Forecast Evaluation

The recursive design uses expanding windows. At each origin, the full Gibbs AM-BVAR-SV, AR(4), and UC-SV are re-estimated using only data available through that origin. Forecasts are evaluated at horizons 1, 2, 3, 4, and 8 quarters ahead.

The AM-BVAR-SV no longer selects a Minnesota lambda by marginal-likelihood grid search. Instead, the adaptive own-lag and cross-lag shrinkage parameters, `kappa_1` and `kappa_2`, are sampled inside the Gibbs sampler and summarized by origin.


In [ ]:
recursive_forecasts_path = OUTPUT_DIR / "recursive_forecasts.csv"
recursive_hyperparameters_path = OUTPUT_DIR / "recursive_am_bvar_sv_hyperparameter_summary.csv"
rmse_path = OUTPUT_DIR / "recursive_rmse_table.csv"

if RUN_RECURSIVE_EVALUATION:
    outputs_exist = recursive_forecasts_path.exists() and recursive_hyperparameters_path.exists()
    if outputs_exist and not FORCE_RECOMPUTE_RECURSIVE:
        recursive_forecasts = pd.read_csv(recursive_forecasts_path)
        bvar_hyperparameters = pd.read_csv(recursive_hyperparameters_path)
    else:
        recursive_forecasts, bvar_hyperparameters = run_recursive_evaluation(
            common_growth,
            first_origin=FIRST_FORECAST_ORIGIN,
            horizons=HORIZONS,
            seed=RANDOM_SEED,
            bvar_n_iter=AM_BVAR_N_ITER_RECURSIVE,
            bvar_burn=AM_BVAR_BURN_RECURSIVE,
            bvar_thin=AM_BVAR_THIN_RECURSIVE,
            bvar_trajectories_per_draw=AM_BVAR_TRAJECTORIES_RECURSIVE,
            ucsv_particles=UCSV_PARTICLES_RECURSIVE,
            ucsv_draws=UCSV_DRAWS_RECURSIVE,
            progress=True,
        )
        recursive_forecasts.to_csv(recursive_forecasts_path, index=False)
        bvar_hyperparameters.to_csv(recursive_hyperparameters_path, index=False)

    rmse_table = compute_rmse_table(recursive_forecasts)
    rmse_table.to_csv(rmse_path, index=False)
    print("Recursive forecasts saved to:", recursive_forecasts_path)
    print("AM-BVAR-SV Gibbs hyperparameter summary saved to:", recursive_hyperparameters_path)
    print("RMSE table saved to:", rmse_path)
    display(rmse_table)
    display(bvar_hyperparameters.tail())
else:
    print("RUN_RECURSIVE_EVALUATION is False; recursive RMSEs were not recomputed.")


## 4. Index Reconstruction Check

Forecast accuracy is evaluated on annualized quarterly log changes. For the fit check, the growth forecasts are recursively back-transformed into index levels and then converted back into implied growth rates. The reconstruction error should be essentially numerical rounding error.

In [ ]:
if RUN_RECURSIVE_EVALUATION and "recursive_forecasts" in globals():
    recursive_index_check = reconstruct_recursive_index_paths(index_levels, recursive_forecasts)
    recursive_index_check.to_csv(OUTPUT_DIR / "recursive_index_reconstruction_check.csv", index=False)
    max_abs_error = recursive_index_check["growth_reconstruction_error"].abs().max()
    print("Max absolute reconstruction error:", max_abs_error)
    display(recursive_index_check.head())
else:
    print("Recursive index reconstruction skipped because recursive forecasts are unavailable.")

## 5. Final Full-Sample Forecast

The final projection fits each model through the latest common complete quarter across all five transformed series. If the latest common quarter is 2026Q1, the forecast quarters are 2026Q2 through 2027Q4.

In [ ]:
final_result = fit_final_forecasts(
    common_growth,
    index_levels,
    seed=RANDOM_SEED,
    forecast_horizon=FINAL_FORECAST_HORIZON,
    bvar_n_iter=AM_BVAR_N_ITER_FINAL,
    bvar_burn=AM_BVAR_BURN_FINAL,
    bvar_thin=AM_BVAR_THIN_FINAL,
    bvar_trajectories_per_draw=AM_BVAR_TRAJECTORIES_FINAL,
    ucsv_particles=UCSV_PARTICLES_FINAL,
    ucsv_draws=UCSV_DRAWS_FINAL,
)

final_bvar_summary = pd.DataFrame([
    summarize_am_bvar_sv_draws(
        final_result["bvar_model"],
        origin=final_result["latest_common_quarter"],
        model_name="AM-BVAR-SV",
        model_system="five_variable_full_sample",
        trajectories_per_draw=AM_BVAR_TRAJECTORIES_FINAL,
    )
])
final_bvar_summary.to_csv(OUTPUT_DIR / "final_am_bvar_sv_hyperparameter_summary.csv", index=False)

print("Final estimation sample:", final_result["full_train"].index.min(), "to", final_result["full_train"].index.max())
print("Forecast quarters:", ", ".join(str(q) for q in final_result["forecast_quarters"]))
print("Index forecast anchor quarter:", final_result["latest_common_quarter"])
print("Full-sample AM-BVAR-SV Gibbs hyperparameter summary:")
display(final_bvar_summary)

display(pd.Series(
    final_result["anchor_levels"],
    index=[INDEX_LABELS[v] for v in VAR_ORDER],
    name="anchor_index_level",
))


## 6. Final Forecast Tables

The AM-BVAR-SV panel is the main forecast. AR(4) and UC-SV are benchmark projections. Index levels are anchored at the actual latest observed index level and recursively back-transformed from the forecast growth rates.

In [ ]:
final_growth_table, final_index_table, am_rate_intervals, am_index_intervals = build_final_output_tables(final_result)
save_final_output_tables(OUTPUT_DIR, final_growth_table, final_index_table, am_rate_intervals, am_index_intervals)

print("Table 3: Final forecast, annualized QoQ log growth rates")
for model_name in MODEL_NAMES:
    print(); print(model_name)
    panel = final_growth_table.loc[final_growth_table["model"] == model_name]
    display(panel.drop(columns=["model", "value_type"]).set_index("forecast_quarter"))

print("Table 4: Final forecast, implied index levels")
for model_name in MODEL_NAMES:
    print(); print(model_name)
    panel = final_index_table.loc[final_index_table["model"] == model_name]
    display(panel.drop(columns=["model", "value_type"]).set_index("forecast_quarter"))

print("Optional Table 5A: AM-BVAR-SV posterior intervals for annualized QoQ growth rates")
display(am_rate_intervals)
print("Optional Table 5B: AM-BVAR-SV posterior intervals for implied index levels")
display(am_index_intervals)

print("Saved outputs in:", OUTPUT_DIR)

## 7. Full Gibbs AM-BVAR-SV Diagnostics

These diagnostics summarize the full-sample Gibbs run used for the final AM-BVAR-SV forecast. The trace plots focus on the sampled adaptive Minnesota hyperparameters and selected covariance diagnostics. The volatility plots show posterior medians and 68 percent bands for `exp(h_t / 2)`, the latent structural-shock standard deviation.


In [ ]:
trace_df = am_bvar_sv_trace_dataframe(final_result["bvar_model"])
trace_path = OUTPUT_DIR / "final_am_bvar_sv_trace_diagnostics.csv"
trace_df.to_csv(trace_path, index=False)

volatility_df = summarize_am_bvar_sv_volatility(final_result["bvar_model"])
volatility_path = OUTPUT_DIR / "final_am_bvar_sv_volatility_summary.csv"
volatility_df.to_csv(volatility_path, index=False)

trace_columns = ["kappa_1", "kappa_2", "log_likelihood_proxy"]
phi_columns = [c for c in trace_df.columns if c.startswith("phi_")]
if phi_columns:
    trace_columns.extend(phi_columns[:2])

fig, axes = plt.subplots(len(trace_columns), 1, figsize=(11, 2.2 * len(trace_columns)), sharex=True)
axes = np.atleast_1d(axes)
for ax, col in zip(axes, trace_columns):
    ax.plot(trace_df["draw"], trace_df[col], color="#1f77b4", linewidth=0.9)
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel("Retained Gibbs draw")
fig.suptitle("Full-sample AM-BVAR-SV Gibbs traces", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
trace_fig_path = OUTPUT_DIR / "final_am_bvar_sv_trace_diagnostics.png"
fig.savefig(trace_fig_path, dpi=180, bbox_inches="tight")
plt.show()

if not volatility_df.empty:
    plot_vars = VAR_ORDER
    fig, axes = plt.subplots(len(plot_vars), 1, figsize=(11, 2.3 * len(plot_vars)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, var in zip(axes, plot_vars):
        panel = volatility_df.loc[volatility_df["variable"] == var].copy()
        x = pd.PeriodIndex(panel["quarter"], freq="Q").to_timestamp(how="end")
        ax.plot(x, panel["median_volatility"], color="#1f77b4", linewidth=1.6, label="Median")
        ax.fill_between(x, panel["p16_volatility"], panel["p84_volatility"], color="#1f77b4", alpha=0.18, label="68% band")
        ax.set_ylabel(VAR_LABELS[var])
        ax.grid(True, alpha=0.25)
    axes[0].legend(frameon=False, loc="upper left")
    fig.suptitle("Posterior latent structural-shock volatility", fontsize=14, fontweight="bold")
    fig.autofmt_xdate()
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    volatility_fig_path = OUTPUT_DIR / "final_am_bvar_sv_volatility_diagnostics.png"
    fig.savefig(volatility_fig_path, dpi=180, bbox_inches="tight")
    plt.show()
else:
    volatility_fig_path = None
    print("Volatility diagnostics require final_result['bvar_model'] to be fit with store_h_path=True.")

print("Diagnostics saved to:")
print("-", trace_path)
print("-", trace_fig_path)
print("-", volatility_path)
if volatility_fig_path is not None:
    print("-", volatility_fig_path)


## 8. Model Fit and Out-of-Sample Performance Charts

The plots below stay on the same scale used by the models and RMSE tables: annualized QoQ log changes, `400 * log(index_t / index_t-1)`. The top panel compares full-sample fitted values with actual data. The bottom panel compares recursive out-of-sample forecasts with actual data for the selected variable and horizon.

Change `PLOT_VARIABLE` and `PLOT_HORIZON` in the setup cell to produce the same figure for a different series or horizon.


In [ ]:
if "recursive_forecasts" not in globals() and recursive_forecasts_path.exists():
    recursive_forecasts = pd.read_csv(recursive_forecasts_path)
if "rmse_table" not in globals() and rmse_path.exists():
    rmse_table = pd.read_csv(rmse_path)

fitted_values_table = make_fitted_value_table(
    common_growth,
    bvar_model=final_result["bvar_model"],
    seed=RANDOM_SEED,
    ucsv_particles=UCSV_PARTICLES_FIT,
)
fitted_values_table.to_csv(OUTPUT_DIR / "full_sample_fitted_values.csv", index=False)

plot_variable = PLOT_VARIABLE
plot_horizon = PLOT_HORIZON
if plot_variable not in VAR_ORDER:
    raise ValueError(f"PLOT_VARIABLE must be one of {VAR_ORDER}")
if plot_horizon not in HORIZONS:
    raise ValueError(f"PLOT_HORIZON must be one of {HORIZONS}")

fit_data = fitted_values_table.loc[fitted_values_table["variable"] == plot_variable].copy()
fit_data["period"] = pd.PeriodIndex(fit_data["quarter"], freq="Q")
fit_pivot = fit_data.pivot(index="period", columns="model", values="fitted").sort_index()
fit_actual = fit_data.drop_duplicates("period").set_index("period")["actual"].sort_index()

oos_data = recursive_forecasts.loc[
    (recursive_forecasts["variable"] == plot_variable)
    & (recursive_forecasts["horizon"] == plot_horizon)
].dropna(subset=["actual", "forecast"]).copy()
oos_data["period"] = pd.PeriodIndex(oos_data["target_quarter"], freq="Q")
oos_pivot = oos_data.pivot_table(index="period", columns="model", values="forecast", aggfunc="mean").sort_index()
oos_actual = oos_data.drop_duplicates("period").set_index("period")["actual"].sort_index()

rmse_lookup = rmse_table.loc[
    (rmse_table["variable"] == plot_variable) & (rmse_table["horizon"] == plot_horizon)
].set_index("model")["RMSE"]

def period_axis(period_index):
    return pd.PeriodIndex(period_index, freq="Q").to_timestamp(how="end")

model_colors = {
    "AM-BVAR-SV": "#1f77b4",
    "AR(4)": "#ff7f0e",
    "UC-SV": "#2ca02c",
}

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

axes[0].plot(period_axis(fit_actual.index), fit_actual.values, color="black", linewidth=2.2, label="Actual")
for model_name in MODEL_NAMES:
    if model_name in fit_pivot.columns:
        axes[0].plot(
            period_axis(fit_pivot.index),
            fit_pivot[model_name],
            color=model_colors[model_name],
            linewidth=1.5,
            alpha=0.95,
            label=f"{model_name} fitted",
        )
axes[0].set_title(f"{VAR_LABELS[plot_variable]}: full-sample fitted values")
axes[0].set_ylabel("Annualized QoQ log change, pp")
axes[0].axhline(0.0, color="0.65", linewidth=0.8)
axes[0].grid(True, alpha=0.25)
axes[0].legend(ncol=2, frameon=False)

axes[1].plot(period_axis(oos_actual.index), oos_actual.values, color="black", linewidth=2.2, label="Actual")
for model_name in MODEL_NAMES:
    if model_name in oos_pivot.columns:
        rmse_label = f"RMSE {rmse_lookup.loc[model_name]:.2f}" if model_name in rmse_lookup.index else "RMSE n/a"
        axes[1].plot(
            period_axis(oos_pivot.index),
            oos_pivot[model_name],
            color=model_colors[model_name],
            linewidth=1.5,
            alpha=0.95,
            label=f"{model_name} forecast ({rmse_label})",
        )
axes[1].set_title(f"Recursive out-of-sample forecasts, horizon {plot_horizon} quarter(s)")
axes[1].set_ylabel("Annualized QoQ log change, pp")
axes[1].axhline(0.0, color="0.65", linewidth=0.8)
axes[1].grid(True, alpha=0.25)
axes[1].legend(ncol=2, frameon=False)

fig.suptitle("Model fit and out-of-sample performance", fontsize=14, fontweight="bold")
fig.autofmt_xdate()
fig.tight_layout(rect=[0, 0, 1, 0.96])

plot_path = OUTPUT_DIR / f"model_fit_and_oos_performance_{plot_variable}_h{plot_horizon}.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
print("Fitted values saved to:", OUTPUT_DIR / "full_sample_fitted_values.csv")
print("Chart saved to:", plot_path)
plt.show()

## 9. Does Wage Growth Improve Forecast Accuracy?

This extension compares two otherwise identical recursive AM-BVAR-SV systems:

1. A five-variable model with ECI wage growth, services excluding housing, housing, core goods, and aggregate core PCE.
2. A four-variable model that drops ECI wage growth and re-estimates the BVAR-SV on the four inflation variables only.

The evaluation uses the same annualized quarterly log-change target, four lags, adaptive Minnesota prior, stochastic-volatility forecasting design, horizons, and expanding-window forecast origins as the main analysis. The key statistic is `RMSE with wage / RMSE without wage`; values below 1 indicate that wage growth improves forecast accuracy.


In [ ]:
def wage_extension_paths(first_origin):
    suffix = str(pd.Period(first_origin, freq="Q"))
    return {
        "forecasts": OUTPUT_DIR / f"wage_growth_recursive_forecasts_{suffix}.csv",
        "hyperparameters": OUTPUT_DIR / f"wage_extension_full_gibbs_hyperparameter_summary_{suffix}.csv",
        "errors": OUTPUT_DIR / f"wage_growth_forecast_errors_{suffix}.csv",
        "rmse": OUTPUT_DIR / f"wage_growth_rmse_tests_{suffix}.csv",
        "subsamples": OUTPUT_DIR / f"wage_growth_subsample_rmse_tests_{suffix}.csv",
    }


def load_wage_extension_outputs(paths):
    forecasts = pd.read_csv(paths["forecasts"])
    hyperparameters = pd.read_csv(paths["hyperparameters"])
    return forecasts, hyperparameters


def run_or_load_wage_extension(first_origin, *, force=False, progress=True):
    paths = wage_extension_paths(first_origin)
    outputs_exist = paths["forecasts"].exists() and paths["hyperparameters"].exists()
    if outputs_exist and not force:
        forecasts, hyperparameters = load_wage_extension_outputs(paths)
    else:
        forecasts, hyperparameters = run_wage_growth_predictive_power_evaluation(
            common_growth,
            first_origin=pd.Period(first_origin, freq="Q"),
            horizons=HORIZONS,
            seed=RANDOM_SEED,
            bvar_n_iter=WAGE_AM_BVAR_N_ITER_RECURSIVE,
            bvar_burn=WAGE_AM_BVAR_BURN_RECURSIVE,
            bvar_thin=WAGE_AM_BVAR_THIN_RECURSIVE,
            bvar_trajectories_per_draw=WAGE_AM_BVAR_TRAJECTORIES_RECURSIVE,
            progress=progress,
        )
        forecasts.to_csv(paths["forecasts"], index=False)
        hyperparameters.to_csv(paths["hyperparameters"], index=False)

    errors = make_wage_comparison_panel(forecasts)
    rmse_tests = compute_wage_growth_predictive_power_table(forecasts)
    subsample_tests = compute_wage_growth_subsample_tables(forecasts)

    errors.to_csv(paths["errors"], index=False)
    rmse_tests.to_csv(paths["rmse"], index=False)
    subsample_tests.to_csv(paths["subsamples"], index=False)
    return forecasts, hyperparameters, errors, rmse_tests, subsample_tests, paths


if RUN_WAGE_EXTENSION:
    wage_forecasts, wage_hyperparameters, wage_forecast_errors, wage_rmse_table, wage_subsample_table, wage_paths = run_or_load_wage_extension(
        WAGE_EXTENSION_FIRST_ORIGIN,
        force=FORCE_RECOMPUTE_WAGE_EXTENSION,
        progress=True,
    )

    wage_display_cols = [
        "target_variable_name",
        "horizon",
        "rmse_with_wage",
        "rmse_without_wage",
        "relative_rmse_with_vs_without_wage",
        "rmse_gain_from_wage_growth_pct",
        "n_forecast_origins",
        "dmw_one_sided_p_value",
        "cw_one_sided_p_value",
        "conclusion",
    ]
    print("Main wage-growth predictive-power table, first origin", WAGE_EXTENSION_FIRST_ORIGIN)
    display(wage_rmse_table[wage_display_cols])

    print("Subsample results by forecast outcome date")
    display(wage_subsample_table[["sample"] + wage_display_cols])

    print("Full Gibbs hyperparameter summaries for the last few recursive origins")
    display(wage_hyperparameters.tail(8))
else:
    print("RUN_WAGE_EXTENSION is False; wage-growth predictive-power extension was skipped.")

if RUN_WAGE_EXTENSION and RUN_WAGE_ROBUSTNESS_2014:
    wage_robust_forecasts, wage_robust_hyperparameters, wage_robust_errors, wage_robust_rmse_table, wage_robust_subsample_table, wage_robust_paths = run_or_load_wage_extension(
        WAGE_EXTENSION_ROBUSTNESS_ORIGIN,
        force=FORCE_RECOMPUTE_WAGE_EXTENSION,
        progress=True,
    )
    print("Robustness table, first origin", WAGE_EXTENSION_ROBUSTNESS_ORIGIN)
    display(wage_robust_rmse_table[wage_display_cols])


### Wage-Growth Figures

The heatmap summarizes the relative RMSEs. The cumulative squared-error plot shows when the wage model accumulates forecast gains or losses. The forecast-comparison panels show actual inflation against the wage-inclusive and no-wage forecasts at economically meaningful horizons.


In [ ]:
if RUN_WAGE_EXTENSION:
    if "wage_forecasts" not in globals():
        wage_forecasts, wage_hyperparameters = load_wage_extension_outputs(wage_paths)
        wage_rmse_table = pd.read_csv(wage_paths["rmse"])
        wage_forecast_errors = make_wage_comparison_panel(wage_forecasts)

    wage_suffix = str(pd.Period(WAGE_EXTENSION_FIRST_ORIGIN, freq="Q"))

    # Figure 1: Relative RMSE heatmap.
    heatmap_data = wage_rmse_table.pivot(
        index="target_variable_name",
        columns="horizon",
        values="relative_rmse_with_vs_without_wage",
    ).reindex([VAR_LABELS[v] for v in INFLATION_TARGETS])
    heatmap_values = heatmap_data.to_numpy(dtype=float)
    finite_values = heatmap_values[np.isfinite(heatmap_values)]
    spread = max(np.nanmax(np.abs(finite_values - 1.0)), 0.02) if len(finite_values) else 0.02
    norm = TwoSlopeNorm(vmin=1.0 - spread, vcenter=1.0, vmax=1.0 + spread)

    fig, ax = plt.subplots(figsize=(9, 4.8))
    image = ax.imshow(heatmap_values, cmap="RdYlGn_r", norm=norm, aspect="auto")
    ax.set_xticks(np.arange(len(heatmap_data.columns)))
    ax.set_xticklabels([f"h={h}" for h in heatmap_data.columns])
    ax.set_yticks(np.arange(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)
    ax.set_title("Relative RMSE: AM-BVAR-SV with wage / without wage")
    for row in range(heatmap_values.shape[0]):
        for col in range(heatmap_values.shape[1]):
            value = heatmap_values[row, col]
            if np.isfinite(value):
                ax.text(col, row, f"{value:.3f}", ha="center", va="center", color="black", fontsize=9)
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Relative RMSE; below 1 favors wage model")
    fig.tight_layout()
    heatmap_path = OUTPUT_DIR / f"wage_growth_relative_rmse_heatmap_{wage_suffix}.png"
    fig.savefig(heatmap_path, dpi=180, bbox_inches="tight")
    plt.show()

    # Figure 2: Cumulative squared-error difference.
    wage_cumdiff = wage_cumulative_loss_difference(
        wage_forecasts,
        variables=WAGE_FORECAST_PLOT_VARIABLES,
        horizons=WAGE_CUMULATIVE_HORIZONS,
    )
    wage_cumdiff.to_csv(OUTPUT_DIR / f"wage_growth_cumulative_loss_difference_{wage_suffix}.csv", index=False)

    fig, axes = plt.subplots(
        len(WAGE_FORECAST_PLOT_VARIABLES),
        len(WAGE_CUMULATIVE_HORIZONS),
        figsize=(13, 6.8),
        sharex=False,
    )
    axes = np.atleast_2d(axes)
    for row, variable in enumerate(WAGE_FORECAST_PLOT_VARIABLES):
        for col, horizon in enumerate(WAGE_CUMULATIVE_HORIZONS):
            ax = axes[row, col]
            panel = wage_cumdiff.loc[
                (wage_cumdiff["variable"] == variable) & (wage_cumdiff["horizon"] == horizon)
            ].copy()
            if not panel.empty:
                x = pd.PeriodIndex(panel["origin"], freq="Q").to_timestamp(how="end")
                ax.plot(x, panel["cumulative_loss_diff"], color="#1f77b4", linewidth=2.0)
            ax.axhline(0.0, color="0.55", linewidth=0.8)
            ax.set_title(f"{VAR_LABELS[variable]}, h={horizon}")
            ax.set_ylabel("Cumulative SE gain")
            ax.grid(True, alpha=0.25)
    fig.suptitle("Cumulative squared-error difference: no-wage minus wage", fontsize=14, fontweight="bold")
    fig.autofmt_xdate()
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    cumdiff_path = OUTPUT_DIR / f"wage_growth_cumulative_loss_difference_{wage_suffix}.png"
    fig.savefig(cumdiff_path, dpi=180, bbox_inches="tight")
    plt.show()

    # Figure 3: Forecast comparisons for core PCE and services ex-housing at h=4 and h=8.
    comparison = make_wage_comparison_panel(wage_forecasts)
    fig, axes = plt.subplots(
        len(WAGE_FORECAST_PLOT_VARIABLES),
        len(WAGE_FORECAST_PLOT_HORIZONS),
        figsize=(13, 6.8),
        sharex=False,
    )
    axes = np.atleast_2d(axes)
    for row, variable in enumerate(WAGE_FORECAST_PLOT_VARIABLES):
        for col, horizon in enumerate(WAGE_FORECAST_PLOT_HORIZONS):
            ax = axes[row, col]
            panel = comparison.loc[
                (comparison["variable"] == variable) & (comparison["horizon"] == horizon)
            ].sort_values("target_period")
            if not panel.empty:
                x = pd.PeriodIndex(panel["target_quarter"], freq="Q").to_timestamp(how="end")
                ax.plot(x, panel["actual"], color="black", linewidth=2.2, label="Actual")
                ax.plot(x, panel[WAGE_MODEL_NAME], color="#1f77b4", linewidth=1.7, label="With wage")
                ax.plot(x, panel[NO_WAGE_MODEL_NAME], color="#d62728", linewidth=1.7, label="Without wage")
            ax.axhline(0.0, color="0.65", linewidth=0.8)
            ax.set_title(f"{VAR_LABELS[variable]}, h={horizon}")
            ax.set_ylabel("Annualized QoQ log change, pp")
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
    fig.suptitle("Forecast comparison: wage-inclusive versus no-wage AM-BVAR-SV", fontsize=14, fontweight="bold")
    fig.autofmt_xdate()
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    comparison_path = OUTPUT_DIR / f"wage_growth_forecast_comparison_{wage_suffix}.png"
    fig.savefig(comparison_path, dpi=180, bbox_inches="tight")
    plt.show()

    print("Wage-growth figures saved to:")
    print("-", heatmap_path)
    print("-", cumdiff_path)
    print("-", comparison_path)
else:
    print("Wage-growth figures skipped because RUN_WAGE_EXTENSION is False.")


## Output Files

This notebook saves:

- `data/raw_index_levels.csv`
- `data/transformed_growth_rates.csv`
- `outputs/data_definitions.csv`
- `outputs/recursive_forecasts.csv`
- `outputs/recursive_am_bvar_sv_hyperparameter_summary.csv`
- `outputs/recursive_rmse_table.csv`
- `outputs/recursive_index_reconstruction_check.csv`
- `outputs/final_am_bvar_sv_hyperparameter_summary.csv`
- `outputs/final_am_bvar_sv_trace_diagnostics.csv`
- `outputs/final_am_bvar_sv_trace_diagnostics.png`
- `outputs/final_am_bvar_sv_volatility_summary.csv`
- `outputs/final_am_bvar_sv_volatility_diagnostics.png`
- `outputs/full_sample_fitted_values.csv`
- `outputs/model_fit_and_oos_performance_core_pce_h1.png` by default; the filename changes when `PLOT_VARIABLE` or `PLOT_HORIZON` changes
- `outputs/wage_growth_recursive_forecasts_2012Q4.csv`
- `outputs/wage_extension_full_gibbs_hyperparameter_summary_2012Q4.csv`
- `outputs/wage_growth_forecast_errors_2012Q4.csv`
- `outputs/wage_growth_rmse_tests_2012Q4.csv`
- `outputs/wage_growth_subsample_rmse_tests_2012Q4.csv`
- `outputs/wage_growth_relative_rmse_heatmap_2012Q4.png`
- `outputs/wage_growth_cumulative_loss_difference_2012Q4.csv`
- `outputs/wage_growth_cumulative_loss_difference_2012Q4.png`
- `outputs/wage_growth_forecast_comparison_2012Q4.png`
- `outputs/wage_growth_recursive_forecasts_2014Q4.csv` when `RUN_WAGE_ROBUSTNESS_2014` is enabled
- `outputs/wage_extension_full_gibbs_hyperparameter_summary_2014Q4.csv` when `RUN_WAGE_ROBUSTNESS_2014` is enabled
- `outputs/wage_growth_forecast_errors_2014Q4.csv` when `RUN_WAGE_ROBUSTNESS_2014` is enabled
- `outputs/wage_growth_rmse_tests_2014Q4.csv` when `RUN_WAGE_ROBUSTNESS_2014` is enabled
- `outputs/wage_growth_subsample_rmse_tests_2014Q4.csv` when `RUN_WAGE_ROBUSTNESS_2014` is enabled
- `outputs/final_forecast_annualized_qoq_growth_rates.csv`
- `outputs/final_forecast_implied_index_levels.csv`
- `outputs/am_bvar_sv_rate_intervals.csv`
- `outputs/am_bvar_sv_index_intervals.csv`

The RMSE evaluations are on annualized quarterly log changes. Values of `relative_rmse_with_vs_without_wage` below 1 mean the wage-inclusive BVAR lowers RMSE relative to the no-wage BVAR. The index-level tables are for interpretation and are anchored at the latest actual observed index level.
